# Stereo-seq (FF footpad tumors): cell type distribution (Fig. 3B; Supplementary Fig. S7C)
Groups: Mattress, Running, YAP5SA; n = 3 per group.
Representative slides in Fig. 3B: FF_Mattress_1, FF_Running_3, FF_YAP5SA_2.
The other six slides are shown in Supplementary Fig. S7C.

In [ ]:
import os
import re
import importlib.metadata
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import decoupler as dc
from scipy import stats

from spatial_utils import *

%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3
sc.set_figure_params(dpi = 100, dpi_save = 300, transparent=True, color_map = 'OrRd')
os.makedirs('./figures', exist_ok=True)

for package in ['stereopy', 'numpy', 'pandas', 'matplotlib', 'scanpy', 'anndata', 'squidpy', 'decoupler']:
    try:
        print(f"{package} version: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed")

# Annotated bin30 objects: <GEO library name>_bin30_annotated.h5ad
SPATIAL_DIR = '../data/stereoseq_annotated'
sample_list = SAMPLE_LIST

In [ ]:
adataL = load_margin_lognorm(SPATIAL_DIR, sample_list)
lognorm4 = load_scored(SPATIAL_DIR, sample_list)

## Whole-section cell type maps (overview, all slides)

In [ ]:
layers_to_view = ['-150um~-100um', '-100um~-50um', '-50um~0um', '0um~50um', '50um~100um', '100um~150um', '150um~200um', '200um~250um', '250um~300um', '300um~350um', '350um~400um', '400um~450um', '450um~500um', '500um~550um', '550um~600um', '600um~650um', '650um~700um', '700um~750um', '750um~800um', '800um~850um', '850um~900um', '900um~950um', '950um~1000um', '1000um~1050um', '1050um~1100um', '1100um~1150um', '1150um~1200um', '1200um~1250um', '1250um~1300um', '1300um~1350um', '1350um~1400um', '1400um~1450um', '1450um~1500um']

In [ ]:
for sample in sample_list:
    bdata = adataL[sample]
    bdata = bdata[bdata.obs['tumor_layer'].isin(layers_to_view)]
    sc.pl.spatial(bdata, color = 'cell_type', spot_size=50, save=f'{sample}_overview.pdf')

## Fig. 3B — Representative slides (high resolution)

In [ ]:
sc.set_figure_params(dpi=1200, color_map = 'OrRd', facecolor=None, transparent=True)

In [ ]:
for sample in sample_list:
    if sample in ['FF_Mattress_1', 'FF_Running_3', 'FF_YAP5SA_2']:
        bdata = adataL[sample]
        bdata = bdata[bdata.obs['tumor_layer'].isin(layers_to_view)]
        sc.pl.spatial(bdata, color = 'cell_type', spot_size=30, frameon=False, title = f'{sample}', save = f'{sample}_high_resolution.svg')

## Supplementary Fig. S7C — Remaining biological replicates

In [ ]:
sc.set_figure_params(dpi=1200, color_map = 'OrRd', facecolor=None, transparent=True)

In [ ]:
for sample in sample_list:
    if sample not in ['FF_Mattress_1', 'FF_Running_3', 'FF_YAP5SA_2']:
        bdata = adataL[sample]
        bdata = bdata[bdata.obs['tumor_layer'].isin(layers_to_view)]
        sc.pl.spatial(bdata, color = 'cell_type', spot_size=30, frameon=False, title = f'{sample}')

In [ ]:
sc.set_figure_params(dpi=300, color_map = 'OrRd', facecolor=None, transparent=True)

## Fig. 3B — Zoom-in of the tumor margin

In [ ]:
a = adataL['FF_Mattress_1']
a = a[a.obs['tumor_layer'].isin(layers_to_view)]

In [ ]:
a.obs['x'] = list(a.obsm['spatial'][:,0])

In [ ]:
a.obs['y'] = list(a.obsm['spatial'][:,1])

In [ ]:
a_1 = a[(a.obs['x'] >= 8000) & (a.obs['x'] <= 10000) & (a.obs['y'] >= 12500) & (a.obs['y'] <= 13000)]

In [ ]:
sc.pl.spatial(a_1, color = 'cell_type', spot_size=30, frameon=False, title = '', save='FF_Mattress_1_margin_image.pdf')

In [ ]:
b = adataL['FF_Running_3']
b = b[b.obs['tumor_layer'].isin(layers_to_view)]

In [ ]:
b.obs['x'] = list(b.obsm['spatial'][:,0])

In [ ]:
b.obs['y'] = list(b.obsm['spatial'][:,1])

In [ ]:
b_1 = b[(b.obs['x'] >= 14000) & (b.obs['x'] <= 16000) & (b.obs['y'] >= 12200) & (b.obs['y'] <= 12700)]

In [ ]:
sc.pl.spatial(b_1, color = 'cell_type', spot_size=30, frameon=False, title = '', save='FF_Running_3_margin_image.pdf')

In [ ]:
c = adataL['FF_YAP5SA_2']
c = c[c.obs['tumor_layer'].isin(layers_to_view)]

In [ ]:
c.obs['x'] = list(c.obsm['spatial'][:,0])

In [ ]:
c.obs['y'] = list(c.obsm['spatial'][:,1])

In [ ]:
c_1 = c[(c.obs['x'] >= 7400) & (c.obs['x'] <= 9400) & (c.obs['y'] >= 11400) & (c.obs['y'] <= 11900)]

In [ ]:
sc.pl.spatial(c_1, color = 'cell_type', spot_size=30, frameon=False, title = '', save='FF_YAP5SA_2_margin_image.pdf')

## Fig. 3B — Cell type proportion across tumor layers
Peritumor: -150–0 µm, margin: 0–150 µm, center: >150 µm from the tumor border.

In [ ]:
# 사용 예시:
combined_adata = combine_adatas(lognorm4, sample_list)

In [ ]:
plot_combined_group_cell_type_proportions(combined_adata)